**<div dir = 'rtl' style="font-family: 'B Nazanin', 'IRANSans', 'Tahoma'; font-size: 40px; line-height: 1.8; text-align: justify;">چند  مثال عملی </div>**

<div dir = 'rtl' style="font-family: 'B Nazanin', 'IRANSans', 'Tahoma'; font-size: 25px; line-height: 1.8; text-align: justify;">امید طالبی  سینا ملاابراهیمی</div>


# Cell 1 : Install Dependencies (Google Colab)

In [1]:

# Segment Anything
!pip -q install git+https://github.com/facebookresearch/segment-anything.git

# Hugging Face
!pip -q install -U transformers accelerate huggingface_hub

# Computer Vision
!pip -q install opencv-python pillow matplotlib supervision

# Utilities
!pip -q install tqdm scipy ipywidgets

# Optional
!pip -q install pycocotools

# Enable widgets
try:
    !jupyter nbextension enable --py widgetsnbextension --sys-prefix > /dev/null 2>&1
except:
    pass

print("=" * 60)
print(" Installation Completed")
print("=" * 60)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 28.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 68.2 MB/s eta 0:00:00
 Installation Completed


# Cell 2 : Environment Check


In [2]:

import torch
import cv2
import matplotlib
import transformers
import numpy as np

print("="*60)
print("Environment")
print("="*60)

print("Torch          :", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))

print("OpenCV         :", cv2.__version__)
print("Transformers   :", transformers.__version__)
print("Matplotlib     :", matplotlib.__version__)

print("="*60)

Environment
Torch          : 2.11.0+cu128
CUDA Available : True
GPU            : Tesla T4
OpenCV         : 4.13.0
Transformers   : 5.13.1
Matplotlib     : 3.10.0


# Cell 3 : Imports


In [3]:

import os
import cv2
import torch
import random
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

from segment_anything import (
    sam_model_registry,
    SamPredictor,
    SamAutomaticMaskGenerator
)

from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device :", DEVICE)

Device : cuda


# Cell 4 : Load Models


In [4]:
import os
import time
import urllib.request

# ----------------------------------------------------------
# Select SAM Model
# ----------------------------------------------------------

SAM_TYPE = "vit_h"
# vit_b
# vit_l
# vit_h

CHECKPOINTS = {

    "vit_b": (
        "sam_vit_b_01ec64.pth",
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
    ),

    "vit_l": (
        "sam_vit_l_0b3195.pth",
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth"
    ),

    "vit_h": (
        "sam_vit_h_4b8939.pth",
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
    )

}

checkpoint_name, checkpoint_url = CHECKPOINTS[SAM_TYPE]

# ----------------------------------------------------------
# Download checkpoint
# ----------------------------------------------------------

if not os.path.exists(checkpoint_name):

    print("Downloading SAM checkpoint...")

    urllib.request.urlretrieve(
        checkpoint_url,
        checkpoint_name
    )

print("Checkpoint Ready")

# ----------------------------------------------------------
# Load SAM
# ----------------------------------------------------------

start = time.time()

sam = sam_model_registry[SAM_TYPE](
    checkpoint=checkpoint_name
)

sam.to(DEVICE)

sam.eval()

elapsed = time.time() - start

print(f"SAM Loaded ({SAM_TYPE})")

print(f"Loading Time : {elapsed:.2f} sec")

# ----------------------------------------------------------
# Predictor
# ----------------------------------------------------------

predictor = SamPredictor(sam)

# ----------------------------------------------------------
# Automatic Generator
# ----------------------------------------------------------

mask_generator = SamAutomaticMaskGenerator(
    sam
)

# ----------------------------------------------------------
# GroundingDINO
# ----------------------------------------------------------

MODEL_ID = "IDEA-Research/grounding-dino-base"

processor = AutoProcessor.from_pretrained(
    MODEL_ID
)

grounding_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    MODEL_ID
).to(DEVICE)

grounding_model.eval()

print("GroundingDINO Loaded")

print("="*60)
print("Everything Ready!")
print("="*60)

Checkpoint Ready
SAM Loaded (vit_h)
Loading Time : 16.92 sec


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.74k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  933MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1182 [00:00<?, ?it/s]

GroundingDINO Loaded
Everything Ready!


# Cell 5 : Utility Functions


In [5]:
from google.colab import files
import matplotlib.pyplot as plt
import numpy as np
import cv2

# ----------------------------------------------------------
# Upload Image
# ----------------------------------------------------------

def load_image():

    uploaded = files.upload()

    image_path = list(uploaded.keys())[0]

    image = cv2.imread(image_path)

    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    return image


# ----------------------------------------------------------
# Build Predictor
# ----------------------------------------------------------

def get_predictor(image):

    predictor = SamPredictor(sam)

    predictor.set_image(image)

    return predictor


# ----------------------------------------------------------
# Show Image
# ----------------------------------------------------------

def show_image(image, title="Image"):

    plt.figure(figsize=(10,10))

    plt.imshow(image)

    plt.title(title)

    plt.axis("off")

    plt.show()


# ----------------------------------------------------------
# Show Mask
# ----------------------------------------------------------

def show_mask(image, mask, title="Mask"):

    plt.figure(figsize=(10,10))

    plt.imshow(image)

    plt.imshow(mask, alpha=0.55)

    plt.title(title)

    plt.axis("off")

    plt.show()


# ----------------------------------------------------------
# Show Points
# ----------------------------------------------------------

def show_points(image,
                mask,
                points,
                labels,
                title="Point Prompt"):

    plt.figure(figsize=(10,10))

    plt.imshow(image)

    plt.imshow(mask, alpha=0.55)

    for point,label in zip(points,labels):

        if label==1:

            plt.scatter(
                point[0],
                point[1],
                marker="*",
                s=220,
                c="lime"
            )

        else:

            plt.scatter(
                point[0],
                point[1],
                marker="x",
                s=150,
                c="red"
            )

    plt.title(title)

    plt.axis("off")

    plt.show()


print(" Utility Functions Ready")

 Utility Functions Ready


In [8]:
image = load_image()

Saving dog.jpg to dog (1).jpg


# Cell 6 : Point Prompt UI


In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------------- Widgets ----------------

x_widget = widgets.IntText(
    value=300,
    description="X:"
)

y_widget = widgets.IntText(
    value=300,
    description="Y:"
)

segment_button = widgets.Button(
    description="Segment",
    button_style="success",
    icon="check"
)

output = widgets.Output()

# ---------------- Callback ----------------

def point_prompt(_):

    with output:

        clear_output(wait=True)

        predictor = get_predictor(image)

        point = np.array([
            [x_widget.value,
             y_widget.value]
        ])

        label = np.array([1])

        masks, scores, logits = predictor.predict(

            point_coords=point,

            point_labels=label,

            multimask_output=True

        )

        best_mask = masks[np.argmax(scores)]

        plt.figure(figsize=(10,10))

        plt.imshow(image)

        plt.imshow(best_mask, alpha=0.55)

        plt.scatter(
            x_widget.value,
            y_widget.value,
            marker="*",
            color="red",
            s=220
        )

        plt.title(
            f"Score = {scores.max():.3f}"
        )

        plt.axis("off")

        plt.show()

segment_button.on_click(point_prompt)

# ---------------- Display ----------------

display(

    widgets.VBox([

        widgets.HTML("<h3>Point Prompt</h3>"),

        x_widget,

        y_widget,

        segment_button,

        output

    ])

)

# Cell 7 : Bounding Box Prompt UI


In [10]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------------- Widgets ----------------

x1_widget = widgets.IntText(value=100, description="X1")
y1_widget = widgets.IntText(value=100, description="Y1")

x2_widget = widgets.IntText(value=400, description="X2")
y2_widget = widgets.IntText(value=400, description="Y2")

box_button = widgets.Button(
    description="Run Box Prompt",
    button_style="info",
    icon="crop"
)

box_output = widgets.Output()

# ---------------- Callback ----------------

def run_box_prompt(_):

    with box_output:

        clear_output(wait=True)

        predictor = get_predictor(image)

        box = np.array([
            x1_widget.value,
            y1_widget.value,
            x2_widget.value,
            y2_widget.value
        ])

        masks, scores, logits = predictor.predict(
            box=box,
            multimask_output=True
        )

        best_mask = masks[np.argmax(scores)]

        plt.figure(figsize=(10,10))

        plt.imshow(image)
        plt.imshow(best_mask, alpha=0.55)

        ax = plt.gca()

        rect = plt.Rectangle(
            (box[0], box[1]),
            box[2]-box[0],
            box[3]-box[1],
            edgecolor="red",
            linewidth=2,
            fill=False
        )

        ax.add_patch(rect)

        plt.title(
            f"Bounding Box Prompt\nScore={scores.max():.3f}"
        )

        plt.axis("off")

        plt.show()

box_button.on_click(run_box_prompt)

display(
    widgets.VBox([
        widgets.HTML("<h3>Bounding Box Prompt</h3>"),
        x1_widget,
        y1_widget,
        x2_widget,
        y2_widget,
        box_button,
        box_output
    ])
)

# Cell 8 : Multi Point Prompt UI


In [11]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt

# ---------------- Widgets ----------------

positive_widget = widgets.Textarea(
    value="220,430\n300,360\n270,470",
    description="Positive",
    layout=widgets.Layout(width="450px", height="120px")
)

negative_widget = widgets.Textarea(
    value="520,350\n470,220\n600,430",
    description="Negative",
    layout=widgets.Layout(width="450px", height="120px")
)

multi_button = widgets.Button(
    description="Run Multi Point",
    button_style="warning",
    icon="play"
)

multi_output = widgets.Output()

# ---------------- Helper ----------------

def parse_points(text):

    pts = []

    lines = text.strip().split("\n")

    for line in lines:

        if line.strip() == "":
            continue

        x, y = line.split(",")

        pts.append([int(x), int(y)])

    return np.array(pts)


# ---------------- Callback ----------------

def run_multi_prompt(_):

    with multi_output:

        clear_output(wait=True)

        predictor = get_predictor(image)

        positive = parse_points(positive_widget.value)

        negative = parse_points(negative_widget.value)

        points = np.concatenate([positive, negative])

        labels = np.concatenate([
            np.ones(len(positive)),
            np.zeros(len(negative))
        ])

        masks, scores, logits = predictor.predict(

            point_coords=points,

            point_labels=labels,

            multimask_output=True
        )

        best_mask = masks[np.argmax(scores)]

        plt.figure(figsize=(10,10))

        plt.imshow(image)

        plt.imshow(best_mask, alpha=0.55)

        plt.scatter(
            positive[:,0],
            positive[:,1],
            marker="*",
            color="lime",
            s=220,
            label="Positive"
        )

        plt.scatter(
            negative[:,0],
            negative[:,1],
            marker="x",
            color="red",
            s=180,
            label="Negative"
        )

        plt.legend()

        plt.title(
            f"Score = {scores.max():.3f}"
        )

        plt.axis("off")

        plt.show()

multi_button.on_click(run_multi_prompt)

display(
    widgets.VBox([
        widgets.HTML("<h3>Multi Point Prompt</h3>"),
        positive_widget,
        negative_widget,
        multi_button,
        multi_output
    ])
)

# Cell 9 : Automatic Mask Generation UI


In [12]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np

# ---------------- Widgets ----------------

max_masks_widget = widgets.IntSlider(
    value=20,
    min=1,
    max=100,
    step=1,
    description="Show"
)

run_auto_button = widgets.Button(
    description="Generate Masks",
    button_style="success",
    icon="magic"
)

auto_output = widgets.Output()

# ---------------- Callback ----------------

def run_auto_masks(_):

    with auto_output:

        clear_output(wait=True)

        print("Generating masks...")

        masks = mask_generator.generate(image)

        print(f"Total Masks : {len(masks)}")

        masks = sorted(
            masks,
            key=lambda x: x["area"],
            reverse=True
        )

        plt.figure(figsize=(12,12))

        plt.imshow(image)

        ax = plt.gca()

        for mask in masks[:max_masks_widget.value]:

            segmentation = mask["segmentation"]

            color = np.concatenate([
                np.random.random(3),
                [0.45]
            ])

            overlay = np.zeros(
                (*segmentation.shape,4)
            )

            overlay[segmentation] = color

            ax.imshow(overlay)

        plt.title(
            f"Automatic Mask Generation\nShowing {max_masks_widget.value} Largest Masks"
        )

        plt.axis("off")

        plt.show()

run_auto_button.on_click(run_auto_masks)

display(
    widgets.VBox([
        widgets.HTML("<h3>Automatic Mask Generator</h3>"),
        max_masks_widget,
        run_auto_button,
        auto_output
    ])
)

# Cell 10 : Text Prompt (GroundingDINO + SAM)


In [13]:
import ipywidgets as widgets
from IPython.display import display, clear_output

import torch
import numpy as np
import matplotlib.pyplot as plt


# ---------------- Widgets ----------------

text_widget = widgets.Text(
    value="dog",
    description="Object:",
    layout=widgets.Layout(width="400px")
)


threshold_widget = widgets.FloatSlider(
    value=0.35,
    min=0.05,
    max=0.9,
    step=0.05,
    description="Score:"
)


text_button = widgets.Button(
    description="Run Text Prompt",
    button_style="primary",
    icon="search"
)


text_output = widgets.Output()



# ---------------- Callback ----------------

def run_text_prompt(_):

    with text_output:

        clear_output(wait=True)

        prompt = text_widget.value.strip()

        if not prompt:
            print("Please enter a prompt")
            return


        print("Searching for:", prompt)


        # GroundingDINO input

        inputs = processor(
            images=image,
            text=prompt,
            return_tensors="pt"
        )

        inputs = {
            k:v.to(DEVICE)
            for k,v in inputs.items()
        }


        # Prediction

        with torch.no_grad():

            outputs = grounding_model(**inputs)



        # New Transformers API

        results = processor.post_process_grounded_object_detection(
            outputs,
            inputs["input_ids"],
            threshold_widget.value,
            target_sizes=[
                image.shape[:2]
            ]
        )


        boxes = results[0]["boxes"]

        scores = results[0]["scores"]

        labels = results[0]["labels"]



        if len(boxes)==0:

            print("No object detected")

            return



        predictor = get_predictor(image)


        plt.figure(figsize=(10,10))

        plt.imshow(image)


        for box,score,label in zip(
            boxes,
            scores,
            labels
        ):


            box = box.cpu().numpy()


            masks, mask_scores, _ = predictor.predict(
                box=box,
                multimask_output=True
            )


            mask = masks[np.argmax(mask_scores)]


            plt.imshow(
                mask,
                alpha=0.45
            )


            x1,y1,x2,y2 = box


            plt.gca().add_patch(

                plt.Rectangle(
                    (x1,y1),
                    x2-x1,
                    y2-y1,
                    fill=False,
                    linewidth=3,
                    edgecolor="red"
                )

            )


            plt.text(
                x1,
                y1,
                f"{label}\n{score:.2f}",
                color="white",
                fontsize=12,
                bbox=dict(
                    facecolor="red",
                    alpha=0.5
                )
            )



        plt.title(
            f"Text Prompt: {prompt}"
        )

        plt.axis("off")

        plt.show()



text_button.on_click(run_text_prompt)


display(

    widgets.VBox([

        widgets.HTML(
            "<h3>Text Prompt (GroundingDINO + SAM)</h3>"
        ),

        text_widget,

        threshold_widget,

        text_button,

        text_output

    ])

)

# Cell 11 : SAM Click (Live demo)


In [14]:
import gradio as gr
import cv2
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image


In [15]:
def sam_click(image, evt: gr.SelectData):

    predictor = get_predictor(image)

    x = evt.index[0]
    y = evt.index[1]

    point = np.array([[x, y]])
    label = np.array([1])

    masks, scores, _ = predictor.predict(

        point_coords=point,

        point_labels=label,

        multimask_output=True

    )

    best_mask = masks[np.argmax(scores)]

    overlay = image.copy()

    color = np.array([0,255,0],dtype=np.uint8)

    overlay[best_mask] = (
        0.5*overlay[best_mask] +
        0.5*color
    ).astype(np.uint8)

    cv2.circle(

        overlay,

        (x,y),

        6,

        (255,0,0),

        -1

    )

    return overlay

In [16]:
with gr.Blocks() as demo:

    gr.Markdown("# Segment Anything Demo")

    input_image = gr.Image(
        type="numpy",
        label="Click On Image"
    )

    output_image = gr.Image(
        label="SAM Output"
    )

    input_image.select(

        fn=sam_click,

        inputs=input_image,

        outputs=output_image

    )

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://06264ca52508ddbbbb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/di

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://06264ca52508ddbbbb.gradio.live
